# Qwen activation smoke test
Select **Runtime → Change runtime type → T4 GPU**. Upload the repository ZIP and run these cells. No dataset or probes are created. GPU availability is not guaranteed.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()  # Upload ai-safety-capstone.zip
archive = next(name for name in uploaded if name.endswith(".zip"))
with zipfile.ZipFile(archive) as z:
    z.extractall("/content")
%cd /content/ai-safety-capstone

In [ ]:
%pip install -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name(0))

Mount Drive before the model run so outputs persist if the runtime disconnects. Use a fresh run folder each time.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from datetime import datetime, timezone
run_dir = "/content/drive/MyDrive/ai-safety-capstone/smoke_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "src/run_model.py", "--device", "cuda", "--output-dir", run_dir], check=True)

In [ ]:
import json, numpy as np
from pathlib import Path
metadata = json.loads((Path(run_dir) / "metadata.json").read_text())
for info in metadata["activations"].values():
    vector = np.load(Path(run_dir) / info["file"], allow_pickle=False)
    assert list(vector.shape) == info["shape"]
    print(info["file"], vector.shape)
print(metadata["response"])